In [1]:
# 1. Initialization

import matplotlib.pyplot as plt
import importlib
from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    TensorDataset,
    random_split
)
from engine import train_one_epoch, evaluate
import json
import os
import pandas as pd
import numpy as np
import torchvision
from pathlib import Path
from torchvision import datasets
from torchvision.transforms import v2
import torch
import random
from torch import nn
from checkpointing import save_checkpoint, load_checkpoint

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    else:
        return torch.device("cpu")

device = get_device()

print('torch:', torch.__version__)
print('built CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print(f"Selected device: {device}")

torch: 2.13.0+cu130
built CUDA: 13.0
CUDA available: True
device: NVIDIA GeForce RTX 5060 Laptop GPU
Selected device: cuda:0


# Reproducibility :yayy:

In [2]:
# Testing seed_everything() - Creating deterministic environment

from utils import seed_everything

seed_everything(2026)

python_value_1 = random.random()
numpy_value_1 = np.random.rand()
torch_value_1 = torch.rand(1)

seed_everything(2026)

python_value_2 = random.random()
numpy_value_2 = np.random.rand()
torch_value_2 = torch.rand(1)

assert python_value_1 == python_value_2

assert numpy_value_1 == numpy_value_2

assert torch.equal(
    torch_value_1,
    torch_value_2,
)

In [3]:
# Configuration metadata

config = {
    "seed": 2026,
    "batch_size": 128,
    "hidden_features": 128,
    "learning_rate": 0.1
}

run_metadata = {
    "torch_version": torch.__version__,
    "torchvision_version": (
        torchvision.__version__
    ),
    "device": str(get_device()),
    "config": config,
}

print(run_metadata)

{'torch_version': '2.13.0+cu130', 'torchvision_version': '0.28.0+cu130', 'device': 'cuda:0', 'config': {'seed': 2026, 'batch_size': 128, 'hidden_features': 128, 'learning_rate': 0.1}}


## Checkpoint implementation

There are two different checkpoints for different tasks

### 1. Inference-only save
- You only need parameters/buffers
- `model.state_dict()`

### 2. Resume-training checkpoint
- completed_epoch
- model_state_dict
- optimizer_state_dict
- best_val_accuracy
- config

optimizer_state_dict contains:
- momentum buffers
- adaptive moving averages
- parameter-group settings

In [4]:
class FashionMLP(nn.Module):
    def __init__(
        self,
        hidden_features: int = 128,
    ) -> None:
        super().__init__()

        self.flatten = nn.Flatten(
            start_dim=1,
        )

        self.network = nn.Sequential(
            nn.Linear(
                28 * 28,
                hidden_features,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_features,
                10,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = self.flatten(x)
        return self.network(x)

In [5]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

seed_everything(
    config["seed"]
)

model = FashionMLP(
    hidden_features=config[
        "hidden_features"
    ]
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config["learning_rate"],
)

In [6]:
print(model.state_dict().keys())

print(optimizer.state_dict().keys())

odict_keys(['network.0.weight', 'network.0.bias', 'network.2.weight', 'network.2.bias'])
dict_keys(['state', 'param_groups'])


In [6]:
# Defining a very basic model:

class FashionMLP(nn.Module):
    def __init__(
        self,
        hidden_features: int = 128
    ) -> None:
        super().__init__()

        self.flatten = nn.Flatten(
            start_dim = 1,
        )

        self.network = nn.Sequential(
            nn.Linear(
                in_features = 28 * 28,
                out_features = hidden_features
            ),
            nn.ReLU(),
            nn.Linear(
                in_features = hidden_features,
                out_features = 10
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = self.flatten(x)
        logits = self.network(x)
        return logits

In [ ]:
checkpoint_path = Path(
    "checkpoints/round_trip.pt"
)

model = FashionMLP()

learning_rate = 0.1

config = {
    "hidden_features": 128,
    "learning_rate": 0.1
}

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

save_checkpoint(
   checkpoint_path,
    completed_epoch=0,
    model=model,
    optimizer=optimizer,
    best_val_accuracy=0.0,
    config=config,
)

assert checkpoint_path.exists()

In [13]:
restored_model = FashionMLP(
    hidden_features=config[
        "hidden_features"
    ]
).to(device)

restored_optimizer = torch.optim.SGD(
    restored_model.parameters(),
    lr=config["learning_rate"],
)


checkpoint = load_checkpoint(
    checkpoint_path,
    model=restored_model,
    optimizer=restored_optimizer,
    device=device,
)


In [ ]:
assert (checkpoint["completed_epoch"] == 0)
assert (checkpoint["best_val_accuracy"] == 0.0)
assert checkpoint["config"] == config

In [2]:
# Goal: Init - stop training - Save - Load - Continue training.

config = {
    "seed": 2026,
    "batch_size": 128,
    "hidden_features": 128,
    "learning_rate": 0.1,
}

# Loading dataset
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# Deterministic split
split_generator = (
    torch.Generator()
    .manual_seed(config["seed"])
)

train_dataset, val_dataset = random_split(
    training_data,
    lengths=[54_000, 6_000],
    generator=split_generator,
)

# Loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
)

In [17]:
seed_everything(
    config["seed"]
)

# Defining model, opt, loss
model = FashionMLP(
    hidden_features=config[
        "hidden_features"
    ]
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config["learning_rate"],
)

loss_fn = nn.CrossEntropyLoss()

best_val_accuracy = -1.0

history = []


In [ ]:
for epoch_index in range(2):
    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
    )

    val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        loss_fn=loss_fn,
        device=device,
    )

    best_val_accuracy = max(
        best_val_accuracy,
        val_metrics["accuracy"],
    )

    completed_epoch = (
        epoch_index + 1
    )

    history.append(
        {
            "epoch": completed_epoch,
            "train_loss": (
                train_metrics["loss"]
            ),
            "train_accuracy": (
                train_metrics["accuracy"]
            ),
            "val_loss": (
                val_metrics["loss"]
            ),
            "val_accuracy": (
                val_metrics["accuracy"]
            ),
        }
    )

    print(
        completed_epoch,
        train_metrics,
        val_metrics,
    )


1 {'loss': 0.7455307351748148, 'accuracy': 0.7467592592592592} {'loss': 0.5385554060935974, 'accuracy': 0.808}
2 {'loss': 0.5018090257732957, 'accuracy': 0.8249814814814814} {'loss': 0.4926447868347168, 'accuracy': 0.8195}


In [ ]:
# Saving current half-trained model
resume_path = Path(
    "checkpoints/resume_demo.pt"
)

save_checkpoint(
    resume_path,
    completed_epoch=2,
    model=model,
    optimizer=optimizer,
    best_val_accuracy=(
        best_val_accuracy
    ),
    config=config,
)

In [ ]:
# Saving history record too.
Path("results").mkdir(
    parents=True,
    exist_ok=True,
)

with Path("results/pre_restart_history.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        history,
        file,
        indent=2,
    )

In [ ]:
# Reconstruct. Try to not run the previous training loop.

model = FashionMLP(
    hidden_features=128
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.2,
)

checkpoint = load_checkpoint(
    "checkpoints/resume_demo.pt",
    model=model,
    optimizer=optimizer,
    device=device,
)

NameError: name 'config' is not defined